# EO-1 — official chat railroads, every string printed

Kernel **eo1**. Same `EO-1-3B` weights. No second VLM.

`demo.ipynb` mixed their robot batch with the tic-tac-toe prompt and used **example1** as the chat image (256 tokens). Their own text example is **example2.png** only.

This notebook runs their published chat paths and then asks for English subtasks. Each turn prints: messages, prompt string, prompt-from-ids, raw continuation, formatted continuation, full decode.

- **readme** — [EO-Robotics/EO1 README](https://github.com/EO-Robotics/EO1): `apply_chat_template(tokenize=True)` (no `add_generation_prompt`), `max_new_tokens=1024`.
- **vl_eval** — their `experiments/8_vllmeval/vlm/model.py` `generate_inner_transformers`: `tokenize=False`, `add_generation_prompt=True`, `process_vision_info`, temp `0.01`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Markdown
from PIL import Image
from eo1_chat import (
    CHATTY,
    OFFICIAL_TTT,
    EO1Chat,
    user_turn,
    user_turn_two,
)
SHOT = ROOT / "screenshot"
ex1_p = SHOT / "example1.jpg"
ex2_p = SHOT / "example2.png"
if not ex1_p.is_file() or not ex2_p.is_file():
    raise FileNotFoundError("screenshot/ missing official stills. python download_frames.py")
ex1 = Image.open(ex1_p).convert("RGB")
ex2 = Image.open(ex2_p).convert("RGB")
display(Markdown(f"**example1.jpg** {ex1.size} — first-notebook scene / wrist partner"))
display(ex1)
display(Markdown(f"**example2.png** {ex2.size} — official README chat image (tic-tac-toe)"))
display(ex2)
print("OFFICIAL_TTT =", repr(OFFICIAL_TTT))

In [ ]:
play = EO1Chat()
print("loaded", play.weights)
print("has processor.generate?", hasattr(play.processor, "generate"))

## 1. Exact GitHub README text path

`example2.png` + official tic-tac-toe string. `tokenize=True`, no generation prompt. This is what they published.

In [ ]:
play.run(
    "readme / example2 / official tic-tac-toe",
    user_turn(ex2, OFFICIAL_TTT),
    railroad="readme",
)

## 2. Their VL-eval chat path on the same official still

`add_generation_prompt=True` (Qwen / their RoboVQA wrapper). Same tic-tac-toe text and `example2.png`.

In [ ]:
play.run(
    "vl_eval / example2 / official tic-tac-toe",
    user_turn(ex2, OFFICIAL_TTT),
    railroad="vl_eval",
)

## 3. Same VL-eval path on example1

So you can see whether the first notebook’s still was the mute, or the railroad was.

In [ ]:
play.run(
    "vl_eval / example1 / official tic-tac-toe",
    user_turn(ex1, OFFICIAL_TTT),
    railroad="vl_eval",
)

## 4. English-while-acting asks on their official chat still

VL-eval railroad. `example2.png`. These are the questions you actually care about.

In [ ]:
for key, text in CHATTY:
    play.run(
        f"vl_eval / example2 / {key}",
        user_turn(ex2, text),
        railroad="vl_eval",
    )

## 5. Both official stills + English subtask

Head + wrist in one user turn. Not in their README (that chat example is one image). Same weights.

In [ ]:
play.run(
    "vl_eval / example1+example2 / next subtask",
    user_turn_two(
        ex1,
        ex2,
        "You see both cameras. In one English sentence, what is the next subtask? Then say I will now …",
    ),
    railroad="vl_eval",
)